# Train SwinUNETR on blob pseudo-labels

Fine-tune a **full SwinUNETR** (encoder + decoder) for binary synapse
segmentation on 2D 128×128 fluorescence patches. The encoder is
initialised from a SimMIM+VICReg pretrained checkpoint; the decoder
starts from random weights.

**Supervision**: blob-based pseudo-labels generated by the pipeline
in `pseudolabels.blobs` (LoG + Meijering + co-localisation +
z-score, see `pseudolabels/blob_pseudolabels.ipynb` for details).

**Loss**: Dice + BCE (standard medical-image segmentation objective).

**Full-image inference**: After training, we demonstrate sliding-window
inference on a reassembled full-resolution image, stitching patch
predictions with overlap averaging.


## Imports


In [ ]:
import os, sys, json, time
from datetime import datetime
from pathlib import Path


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt


## Path setup


In [ ]:
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / 'root'
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('REPO_ROOT:', REPO_ROOT)
print('ROOT     :', ROOT)

In [ ]:
# Training infrastructure
from synaptic_ssl.training.config import BaseCfg, DataCfg, ModelCfg, dump_config
from synaptic_ssl.training.seeding import seed_everything
from synaptic_ssl.training.logging import setup_logger, CSVMetricLogger
from synaptic_ssl.training.data import compute_channel_stats
from synaptic_ssl.training.lr_schedule import param_groups_layer_decay, make_warmup_cosine
from synaptic_ssl.training.checkpoints import save_checkpoint, load_checkpoint, find_latest_checkpoint
from synaptic_ssl.pseudolabels.blobs import (
    BlobPseudoCfg, generate_blob_pseudolabel,
    compute_global_meijering_threshold,
    generate_pseudolabels_fullimage,
)
from synaptic_ssl.ssl_training.post_training import build_run_label

# Segmentation-specific
from synaptic_ssl.segmentation import (
    SegTrainCfg,
    PseudoLabelSegDataset,
    DiceBCELoss, compute_dice_metric,
    build_swinunetr, load_pretrained_encoder_into_swinunetr, count_params,
    SegTrainTransform, SegValTransform,
    sliding_window_predict, predict_full_image,
    plot_seg_overlay, plot_seg_comparison, plot_seg_curves,
    plot_full_image_result,
)


## Configuration


In [ ]:
base_cfg = BaseCfg(
    seed             = 42,
    output_root      = '../outputs',
    experiment_name  = 'swinunetr_seg_pseudolabels',
    tag              = 'pretrained_enc',
    method_name      = 'swinunetr_seg',
    init_source      = 'local_ckpt',
    pretrained_ckpt_path = None,  # <-- SET THIS to your pretrained encoder .pt
    resume_path      = None,
    dry_run          = False,
)


In [ ]:
data_cfg = DataCfg(
    data_root        = '../../data/patches_128',
    exclude_patterns = ['KONTROLA'],
    val_split        = 0.15,
    batch_size       = 16,
    num_workers      = 2,
    pin_memory       = True,
    channel_names    = ['pre_synaptic', 'post_synaptic', 'structural'],
)


In [ ]:
model_cfg = ModelCfg(
    in_channels    = 3,
    img_size       = 128,
    feature_size   = 96,
    patch_size     = 2,
    window_size    = 7,
    depths         = (2, 2, 6, 2),
    num_heads      = (3, 6, 12, 24),
    dropout_path_rate = 0.1,
)


In [ ]:
seg_cfg = SegTrainCfg(
    epochs                = 100,
    warmup_epochs         = 5,
    base_lr               = 1e-4,
    decoder_lr            = 5e-4,
    weight_decay          = 0.05,
    layer_decay           = 0.75,
    grad_clip_norm        = 5.0,
    freeze_encoder_epochs = 3,
    save_every_n_epochs   = 10,
    val_metric_key        = 'dice',
    val_metric_direction  = 'max',
    model_save_name       = 'swinunetr_seg_best.pt',
    dice_weight           = 1.0,
    bce_weight            = 1.0,
    dice_smooth           = 1.0,
    pseudolabel_cache_dir = '../outputs/pseudolabel_cache',
)


In [ ]:
pseudo_cfg = BlobPseudoCfg(
    log_min_sigma    = 0.7,
    log_max_sigma    = 1.8,
    log_num_sigma    = 5,
    log_threshold    = 0.005,
    coloc_dilation   = 2,
    dendrite_threshold = None,  # computed below from data
    use_zscore       = True,
    zscore_threshold = 5.0,
)


## Output directory and logger


In [ ]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(base_cfg.output_root) / f'{base_cfg.experiment_name}_{base_cfg.tag}_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)
print('save_dir =', save_dir)


In [ ]:
logger = setup_logger('seg', save_dir / 'run.log')
logger.info(f'experiment = {base_cfg.experiment_name}')
logger.info(f'tag        = {base_cfg.tag}')
logger.info(f'method     = {base_cfg.method_name}')
logger.info(f'init_source= {base_cfg.init_source}')
logger.info(f'save_dir   = {save_dir}')


In [ ]:
dump_config(
    save_dir / 'config.json',
    base=base_cfg, data=data_cfg, model=model_cfg, seg=seg_cfg, pseudo=pseudo_cfg,
)
logger.info('config.json written')


## Seed and device


In [ ]:
generator = seed_everything(base_cfg.seed)
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'seed   = {base_cfg.seed}')
logger.info(f'device = {device}')
if device.type == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f'gpu    = {name}  ({mem:.1f} GB)')


## Data


In [ ]:
# Raw patch dataset
from data.patch_dataset import PatchDataset
raw_dataset = PatchDataset(root=data_cfg.data_root, exclude_patterns=data_cfg.exclude_patterns)
logger.info(f'raw patches = {len(raw_dataset)}')
sample = raw_dataset[0]
logger.info(f'sample shape = {tuple(sample.shape)}  dtype = {sample.dtype}')
assert sample.ndim == 3 and sample.shape[0] == model_cfg.in_channels
assert sample.shape[-1] == model_cfg.img_size


In [ ]:
# Channel stats (computed on the train split)
n_val   = int(len(raw_dataset) * data_cfg.val_split)
n_train = len(raw_dataset) - n_val
train_subset, val_subset = random_split(raw_dataset, [n_train, n_val], generator=generator)
logger.info(f'split: train={n_train}  val={n_val}')

ch_mean, ch_std = compute_channel_stats(
    train_subset, in_channels=model_cfg.in_channels,
    max_samples=data_cfg.channel_stats_max_samples,
)
for name, m, s in zip(data_cfg.channel_names, ch_mean.tolist(), ch_std.tolist()):
    logger.info(f'  ch[{name:>14s}] mean={m:.5f}  std={s:.5f}')
stats = {'channel_names': list(data_cfg.channel_names), 'mean': ch_mean.tolist(), 'std': ch_std.tolist()}
(save_dir / 'channel_stats.json').write_text(json.dumps(stats, indent=2))


### Compute global Meijering threshold for pseudo-labels

The dendrite mask uses a Meijering ridge filter; its threshold
must be computed globally across a sample of patches (per-patch
Otsu is unreliable). We sample structural-channel patches from
the training split.


In [ ]:
# Sample structural-channel patches for global threshold
n_threshold_samples = min(200, len(train_subset))
struct_patches = []
for i in range(n_threshold_samples):
    patch_t = train_subset[i]
    struct_patches.append(patch_t[pseudo_cfg.structural_channel].numpy())

global_meijering_thr = compute_global_meijering_threshold(
    struct_patches,
    sigmas=pseudo_cfg.dendrite_sigmas,
    method='otsu',
)
pseudo_cfg.dendrite_threshold = global_meijering_thr
logger.info(f'global Meijering threshold = {global_meijering_thr:.6f}')


### Build segmentation datasets with pseudo-labels

Pseudo-labels are generated in **full-image mode**: the Meijering
dendrite filter and soma detector run on the full reassembled image
(no border artifacts, intact somas), then the structural mask is
sliced back. LoG blob detection + z-score + co-localisation run
per-patch (fine at 128×128).


In [ ]:
# Build PatchDataset views for train/val indices
# We need access to the underlying PatchDataset for pseudo-label generation
train_indices = train_subset.indices
val_indices   = val_subset.indices

# Create index-filtered PatchDataset wrappers
class IndexedPatchDataset:
    """Wraps PatchDataset to expose only a subset of indices."""
    def __init__(self, base_ds, indices):
        self.root = base_ds.root
        self.channels = base_ds.channels
        self.records = [base_ds.records[i] for i in indices]
    def __len__(self):
        return len(self.records)

train_patch_ds = IndexedPatchDataset(raw_dataset, train_indices)
val_patch_ds   = IndexedPatchDataset(raw_dataset, val_indices)
logger.info(f'train patches (pseudo-label) = {len(train_patch_ds.records)}')
logger.info(f'val   patches (pseudo-label) = {len(val_patch_ds.records)}')

# --- Full-image pseudo-label generation ---
# Run structural mask (Meijering + soma) on full reassembled images,
# then LoG per patch. This avoids border artifacts on dendrites/somas.
from data.reassemble import reassemble_image, list_image_indices

all_image_indices = list_image_indices(
    data_cfg.data_root, exclude_patterns=data_cfg.exclude_patterns,
)
logger.info(f'generating full-image pseudo-labels for {len(all_image_indices)} images...')

all_masks = {}  # filename -> (H, W) uint8
for img_idx in tqdm(all_image_indices, desc='full-image pseudo-labels'):
    full_img, records = reassemble_image(
        data_cfg.data_root, img_idx,
        exclude_patterns=data_cfg.exclude_patterns,
    )
    masks_for_image = generate_pseudolabels_fullimage(
        full_img, records, pseudo_cfg,
    )
    all_masks.update(masks_for_image)

logger.info(f'total pseudo-label masks = {len(all_masks)}')
n_positive = sum(m.sum() for m in all_masks.values())
logger.info(f'total positive pixels = {n_positive}')


In [ ]:
# Augmentation pipelines
train_transform = SegTrainTransform(ch_mean, ch_std)
val_transform   = SegValTransform(ch_mean, ch_std)


In [ ]:
# Build datasets with precomputed full-image pseudo-labels
train_seg_ds = PseudoLabelSegDataset(
    train_patch_ds, pseudo_cfg,
    cache_dir=seg_cfg.pseudolabel_cache_dir,
    transform=train_transform,
    precomputed_masks=all_masks,
)
val_seg_ds = PseudoLabelSegDataset(
    val_patch_ds, pseudo_cfg,
    cache_dir=seg_cfg.pseudolabel_cache_dir,
    transform=val_transform,
    precomputed_masks=all_masks,
)
logger.info(f'train seg dataset = {len(train_seg_ds)}')
logger.info(f'val   seg dataset = {len(val_seg_ds)}')


In [ ]:
# DataLoaders
common = dict(
    num_workers=data_cfg.num_workers,
    pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
train_loader = DataLoader(train_seg_ds, batch_size=data_cfg.batch_size, shuffle=True,  drop_last=True, **common)
val_loader   = DataLoader(val_seg_ds,   batch_size=data_cfg.batch_size, shuffle=False, **common)
logger.info(f'train batches = {len(train_loader)}  val batches = {len(val_loader)}')


## Sanity checks


In [ ]:
# Verify batch shapes
batch_img, batch_mask = next(iter(train_loader))
assert batch_img.shape == (data_cfg.batch_size, model_cfg.in_channels, model_cfg.img_size, model_cfg.img_size)
assert batch_mask.shape == (data_cfg.batch_size, 1, model_cfg.img_size, model_cfg.img_size)
assert batch_img.dtype == torch.float32
assert batch_mask.dtype == torch.float32
assert torch.isfinite(batch_img).all()
logger.info(f'[ok] batch shape: img={tuple(batch_img.shape)} mask={tuple(batch_mask.shape)}')
logger.info(f'     mask coverage = {batch_mask.mean().item():.4f}')


In [ ]:
# Visualise a few training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(min(4, data_cfg.batch_size)):
    img = batch_img[i, 0].numpy()  # pre-synaptic channel
    msk = batch_mask[i, 0].numpy()
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'Image ch0 [{i}]')
    axes[0, i].axis('off')
    axes[1, i].imshow(msk, cmap='hot', vmin=0, vmax=1)
    axes[1, i].set_title(f'Pseudo-label [{i}]  ({int(msk.sum())} px)')
    axes[1, i].axis('off')
fig.suptitle('Training batch: images (top) and pseudo-labels (bottom)', y=1.02)
fig.tight_layout()
fig.savefig(save_dir / 'sanity_batch.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Build SwinUNETR model
model = build_swinunetr(model_cfg, out_channels=1).to(device)
logger.info(f'SwinUNETR total params = {count_params(model) / 1e6:.2f} M')


In [ ]:
# Load pretrained encoder
if base_cfg.pretrained_ckpt_path is not None:
    enc_summary = load_pretrained_encoder_into_swinunetr(
        model, base_cfg.pretrained_ckpt_path, logger=logger,
    )
    # Use channel stats from the pretrained checkpoint if available
    if enc_summary.get('channel_mean') is not None:
        logger.info(f'  pretrained channel_mean = {enc_summary["channel_mean"]}')
        logger.info(f'  pretrained channel_std  = {enc_summary["channel_std"]}')
else:
    logger.warning('No pretrained_ckpt_path set -- encoder starts from random init')


In [ ]:
# Sanity check: forward pass
model.eval()
with torch.no_grad():
    test_out = model(batch_img[:2].to(device))
assert test_out.shape == (2, 1, model_cfg.img_size, model_cfg.img_size), f'got {test_out.shape}'
logger.info(f'[ok] forward pass: input {tuple(batch_img[:2].shape)} -> output {tuple(test_out.shape)}')
model.train()


In [ ]:
# Sanity check: gradient flow
model.train()
loss_fn = DiceBCELoss(
    dice_weight=seg_cfg.dice_weight,
    bce_weight=seg_cfg.bce_weight,
    smooth=seg_cfg.dice_smooth,
)
_opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
_opt.zero_grad(set_to_none=True)
_img = batch_img[:2].to(device)
_msk = batch_mask[:2].to(device)
_out = model(_img)
_losses = loss_fn(_out, _msk)
_losses['loss'].backward()
n_grad = sum(1 for p in model.parameters() if p.grad is not None)
n_tot  = sum(1 for _ in model.parameters())
_opt.step()
del _opt, _losses, _out, _img, _msk
logger.info(f'[ok] grad flow: {n_grad}/{n_tot} params got gradients')


## Overfit check

Train on a single mini-batch to verify the model can drive loss to ~0.


In [ ]:
RUN_OVERFIT     = True
N_OVERFIT_STEPS = 100
OVERFIT_LR      = 5e-4


In [ ]:
if RUN_OVERFIT:
    # Fresh model for overfit test
    of_model = build_swinunetr(model_cfg, out_channels=1).to(device)
    of_opt = torch.optim.AdamW(of_model.parameters(), lr=OVERFIT_LR)
    of_img = batch_img.to(device)
    of_msk = batch_mask.to(device)

    of_losses_hist = []
    of_dice_hist = []
    of_model.train()
    for step in tqdm(range(N_OVERFIT_STEPS), desc='overfit'):
        of_opt.zero_grad(set_to_none=True)
        out = of_model(of_img)
        losses = loss_fn(out, of_msk)
        losses['loss'].backward()
        of_opt.step()
        of_losses_hist.append(losses['loss'].item())
        with torch.no_grad():
            of_dice_hist.append(compute_dice_metric(out, of_msk).item())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(of_losses_hist)
    ax1.set_xlabel('step'); ax1.set_ylabel('loss'); ax1.set_title('Overfit loss')
    ax1.grid(True, alpha=0.3)
    ax2.plot(of_dice_hist, color='green')
    ax2.set_xlabel('step'); ax2.set_ylabel('Dice'); ax2.set_title('Overfit Dice')
    ax2.grid(True, alpha=0.3)
    fig.suptitle(f'Overfit check: {N_OVERFIT_STEPS} steps on 1 batch')
    fig.tight_layout()
    fig.savefig(save_dir / 'overfit_check.png', dpi=150, bbox_inches='tight')
    plt.show()
    logger.info(f'overfit: final loss={of_losses_hist[-1]:.5f}  dice={of_dice_hist[-1]:.4f}')

    del of_model, of_opt, of_img, of_msk
    torch.cuda.empty_cache() if device.type == 'cuda' else None
else:
    logger.info('overfit check skipped')


## Full training


### Reset for full training


In [ ]:
# Re-seed for deterministic full run
generator = seed_everything(base_cfg.seed)

# Rebuild model and load pretrained encoder
model = build_swinunetr(model_cfg, out_channels=1).to(device)
if base_cfg.pretrained_ckpt_path is not None:
    load_pretrained_encoder_into_swinunetr(
        model, base_cfg.pretrained_ckpt_path, logger=logger,
    )
logger.info(f'[reset] model params = {count_params(model) / 1e6:.2f} M')


In [ ]:
# Separate param groups: encoder (layer-decay LR) + decoder (flat LR)
encoder_groups = param_groups_layer_decay(
    model.swinViT,
    base_lr=seg_cfg.base_lr,
    weight_decay=seg_cfg.weight_decay,
    layer_decay=seg_cfg.layer_decay,
)

# Decoder = everything not in swinViT
encoder_param_ids = {id(p) for p in model.swinViT.parameters()}
decoder_params = [p for p in model.parameters() if id(p) not in encoder_param_ids]
decoder_group = {'params': decoder_params, 'lr': seg_cfg.decoder_lr, 'weight_decay': seg_cfg.weight_decay}

optimizer = torch.optim.AdamW(encoder_groups + [decoder_group], betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, make_warmup_cosine(seg_cfg.warmup_epochs, seg_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == 'cuda')

logger.info(f'encoder param groups = {len(encoder_groups)}')
logger.info(f'decoder params       = {sum(p.numel() for p in decoder_params) / 1e6:.2f} M')


### Resume from checkpoint


In [ ]:
start_epoch     = 1
best_val_metric = float('-inf')  # Dice: higher is better
best_epoch      = 0

_resume = base_cfg.resume_path
if _resume is not None:
    p = Path(_resume)
    if p.is_dir():
        p = find_latest_checkpoint(p)
    if p is None or not Path(p).exists():
        logger.warning(f'resume path {_resume!r} not found -- starting fresh')
    else:
        ckpt = load_checkpoint(
            p, encoder=model,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            map_location=device,
        )
        start_epoch     = int(ckpt.get('epoch', 0)) + 1
        best_val_metric = ckpt.get('val_metric', best_val_metric) or best_val_metric
        best_epoch      = int(ckpt.get('epoch', 0))
        logger.info(f'resumed from {p} (epoch {ckpt.get("epoch")})')
logger.info(f'start_epoch = {start_epoch}  best_val_metric = {best_val_metric}')


### CSV metric logger


In [ ]:
csv_fields = [
    'epoch', 'phase',
    'train_loss', 'train_dice_loss', 'train_bce_loss',
    'val_loss', 'val_dice',
    'lr_encoder', 'lr_decoder',
    'epoch_time_s', 'train_time_s', 'val_time_s',
    'best_val_metric', 'best_epoch',
    'grad_norm_mean', 'grad_norm_max',
]
csv_logger = CSVMetricLogger(save_dir / 'metrics.csv', csv_fields)


### Training-loop helpers


In [ ]:
def freeze_encoder(mdl, freeze):
    for p in mdl.swinViT.parameters():
        p.requires_grad = not freeze

def get_lrs(opt):
    enc_lrs = [g['lr'] for g in opt.param_groups if g.get('stage') is not None]
    dec_lrs = [g['lr'] for g in opt.param_groups if g.get('stage') is None]
    return (max(enc_lrs) if enc_lrs else 0.0,
            dec_lrs[0] if dec_lrs else 0.0)

def all_trainable_params(mdl):
    return [p for p in mdl.parameters() if p.requires_grad]


### Full training loop


In [ ]:
train_losses, val_dices, lr_history = [], [], []
total_t0 = time.time()


In [ ]:
_better = lambda new, best: new > best  # Dice: higher is better

if base_cfg.dry_run:
    logger.info('dry_run=True -- skipping full training loop')
else:
    for epoch in range(start_epoch, seg_cfg.epochs + 1):
        in_warmup = epoch <= seg_cfg.freeze_encoder_epochs
        freeze_encoder(model, in_warmup)
        phase = 'frozen' if in_warmup else 'full'

        # ---- TRAIN ----
        model.train()
        running_loss, running_dice_l, running_bce_l = 0.0, 0.0, 0.0
        grads = []
        t_train = time.time()
        pbar = tqdm(train_loader, desc=f'ep {epoch}/{seg_cfg.epochs} [{phase}]', leave=False)
        for img_b, mask_b in pbar:
            img_b  = img_b.to(device, non_blocking=True)
            mask_b = mask_b.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
                logits = model(img_b)
                losses = loss_fn(logits, mask_b)
            scaler.scale(losses['loss']).backward()
            scaler.unscale_(optimizer)
            gn = torch.nn.utils.clip_grad_norm_(
                all_trainable_params(model), max_norm=seg_cfg.grad_clip_norm,
            )
            grads.append(gn.item())
            scaler.step(optimizer)
            scaler.update()
            running_loss   += losses['loss'].item()
            running_dice_l += losses['dice_loss'].item()
            running_bce_l  += losses['bce_loss'].item()
            pbar.set_postfix(loss=f'{losses["loss"].item():.4f}')
        n_tb = max(1, len(train_loader))
        train_loss     = running_loss / n_tb
        train_dice_l   = running_dice_l / n_tb
        train_bce_l    = running_bce_l / n_tb
        train_time = time.time() - t_train
        scheduler.step()

        # ---- VALIDATE ----
        model.eval()
        val_loss_sum, val_dice_sum, n_vb = 0.0, 0.0, 0
        t_val = time.time()
        with torch.no_grad():
            for img_b, mask_b in val_loader:
                img_b  = img_b.to(device, non_blocking=True)
                mask_b = mask_b.to(device, non_blocking=True)
                with torch.amp.autocast(device.type, enabled=device.type == 'cuda'):
                    logits = model(img_b)
                    v_losses = loss_fn(logits, mask_b)
                val_loss_sum += v_losses['loss'].item()
                val_dice_sum += compute_dice_metric(logits, mask_b).item()
                n_vb += 1
        val_loss = val_loss_sum / max(1, n_vb)
        val_dice = val_dice_sum / max(1, n_vb)
        val_time = time.time() - t_val
        val_metric = val_dice  # Dice is our primary metric

        enc_lr, dec_lr = get_lrs(optimizer)
        epoch_time = train_time + val_time
        gmean = sum(grads) / max(1, len(grads))
        gmax  = max(grads) if grads else 0.0

        # ---- CHECKPOINTING ----
        improved = ''
        if _better(val_metric, best_val_metric):
            best_val_metric = val_metric
            best_epoch      = epoch
            improved        = ' *best*'
            save_checkpoint(
                save_dir / 'best_model.pt',
                encoder=model, heads=None,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
                extra={'channel_mean': ch_mean.tolist(),
                       'channel_std':  ch_std.tolist(),
                       'val_dice': val_dice},
            )
        save_checkpoint(
            save_dir / 'last.pt',
            encoder=model, heads=None,
            optimizer=optimizer, scheduler=scheduler, scaler=scaler,
            epoch=epoch, val_metric=val_metric, train_loss=train_loss,
            extra={'channel_mean': ch_mean.tolist(),
                   'channel_std':  ch_std.tolist()},
        )
        if seg_cfg.save_every_n_epochs and epoch % seg_cfg.save_every_n_epochs == 0:
            save_checkpoint(
                save_dir / f'epoch_{epoch:04d}.pt',
                encoder=model, heads=None,
                optimizer=optimizer, scheduler=scheduler, scaler=scaler,
                epoch=epoch, val_metric=val_metric, train_loss=train_loss,
            )

        # ---- LOGGING ----
        train_losses.append(train_loss)
        val_dices.append(val_dice)
        lr_history.append((enc_lr, dec_lr))
        csv_logger.log(dict(
            epoch=epoch, phase=phase,
            train_loss=train_loss, train_dice_loss=train_dice_l, train_bce_loss=train_bce_l,
            val_loss=val_loss, val_dice=val_dice,
            lr_encoder=enc_lr, lr_decoder=dec_lr,
            epoch_time_s=epoch_time, train_time_s=train_time, val_time_s=val_time,
            best_val_metric=best_val_metric, best_epoch=best_epoch,
            grad_norm_mean=gmean, grad_norm_max=gmax,
        ))
        logger.info(
            f'ep {epoch:3d}/{seg_cfg.epochs} [{phase}]  '
            f'train={train_loss:.5f}  val_loss={val_loss:.5f}  '
            f'dice={val_dice:.4f}  '
            f'lr(enc/dec)={enc_lr:.2e}/{dec_lr:.2e}  t={epoch_time:.1f}s  gn={gmean:.2f}{improved}'
        )
    csv_logger.close()
    logger.info(f'total training time = {(time.time() - total_t0) / 60:.1f} min')


## After training


In [ ]:
# Reload best checkpoint
best_path = save_dir / 'best_model.pt'
if best_path.exists():
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    if 'encoder_state_dict' in ckpt:
        model.load_state_dict(ckpt['encoder_state_dict'])
    logger.info(f'reloaded best model from epoch {ckpt.get("epoch")}, dice={ckpt.get("val_dice", ckpt.get("val_metric")):.4f}')
else:
    logger.warning('no best_model.pt found')


In [ ]:
# Plot training curves
_ = plot_seg_curves(
    save_dir / 'metrics.csv',
    run_label=f'{base_cfg.method_name} | {base_cfg.init_source}',
    save_to=save_dir / 'training_curves.png',
)
plt.show()


In [ ]:
# Visualise predictions on a few val patches
model.eval()
vis_batch_img, vis_batch_mask = next(iter(val_loader))
with torch.no_grad():
    vis_logits = model(vis_batch_img.to(device))
    vis_probs  = torch.sigmoid(vis_logits).cpu()

for i in range(min(4, len(vis_batch_img))):
    fig = plot_seg_overlay(
        vis_batch_img[i], vis_batch_mask[i], vis_probs[i],
        channel=0, title=f'Val patch {i}',
        save_to=save_dir / f'val_pred_{i}.png',
    )
    plt.show()


## Full-image sliding-window inference

Training is on 128×128 patches. For true detection on a full image
we reassemble the original (untiled) image and run the model with a
**sliding window** that moves across the image with overlap. The
overlapping predictions are averaged to produce a smooth probability
map.


In [ ]:
from data.reassemble import reassemble_image as _ri, list_image_indices as _li
from synaptic_ssl.pseudolabels.blobs import generate_pseudolabels_fullimage

# List available full images
available_images = _li(
    data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
logger.info(f'available full images: {available_images[:10]}...')


In [ ]:
# Pick the first available image for demonstration
demo_image_idx = available_images[0]
logger.info(f'demo image index = {demo_image_idx}')

full_image, full_records = reassemble_image(
    data_cfg.data_root, demo_image_idx,
    exclude_patterns=data_cfg.exclude_patterns,
)
logger.info(f'full image shape = {full_image.shape}')


In [ ]:
# Run sliding-window inference
model.eval()
prob_map = sliding_window_predict(
    model,
    full_image,
    patch_size=model_cfg.img_size,
    ch_mean=ch_mean,
    ch_std=ch_std,
    device=device,
    overlap=0.5,
    batch_size=16,
)
logger.info(f'prob_map shape = {prob_map.shape}  range = [{prob_map.min():.3f}, {prob_map.max():.3f}]')


In [ ]:
# Generate pseudo-label for the full image using full-image mode
# (structural mask on the full picture, LoG per patch)
demo_masks = generate_pseudolabels_fullimage(
    full_image, full_records, pseudo_cfg,
)

# Reassemble pseudo-labels into the full-image grid
patch_size = int(full_records[0]['patch_size'])
full_pseudo = np.zeros(full_image.shape[1:], dtype=np.float32)
for rec in full_records:
    r, c = int(rec['grid_row']), int(rec['grid_col'])
    y0, x0 = r * patch_size, c * patch_size
    full_pseudo[y0:y0+patch_size, x0:x0+patch_size] = demo_masks[rec['filename']].astype(np.float32)

logger.info(f'full pseudo-label: {int(full_pseudo.sum())} positive pixels')


In [ ]:
# Visualise full-image result
fig = plot_full_image_result(
    full_image, prob_map, pseudolabel=full_pseudo,
    channel=0, threshold=0.5,
    title=f'Full image {demo_image_idx}: sliding-window inference (overlap=0.5)',
    save_to=save_dir / 'full_image_inference.png',
)
plt.show()


In [ ]:
# Detailed comparison panel
fig = plot_seg_comparison(
    full_image, full_pseudo, prob_map,
    channel_names=data_cfg.channel_names,
    threshold=0.5,
    title=f'Full image {demo_image_idx}: channels + pseudo-label + prediction',
    save_to=save_dir / 'full_image_comparison.png',
)
plt.show()


In [ ]:
# Compute Dice on the full image
pred_binary = (prob_map > 0.5).astype(np.float32)
intersection = (pred_binary * full_pseudo).sum()
full_dice = (2 * intersection + 1e-6) / (pred_binary.sum() + full_pseudo.sum() + 1e-6)
logger.info(f'Full-image Dice (vs pseudo-label) = {full_dice:.4f}')
logger.info(f'Pred positive px = {int(pred_binary.sum())}  Pseudo positive px = {int(full_pseudo.sum())}')


## Save model


In [ ]:
if best_path.exists():
    src = torch.load(best_path, map_location=device, weights_only=False)
    model_path = save_dir / seg_cfg.model_save_name
    torch.save({
        'model_state_dict':   src['encoder_state_dict'],
        'channel_mean':       src.get('channel_mean', ch_mean.tolist()),
        'channel_std':        src.get('channel_std',  ch_std.tolist()),
        'epoch':              src.get('epoch'),
        'val_metric':         src.get('val_metric'),
        'val_dice':           src.get('val_dice'),
        'init_source':        base_cfg.init_source,
        'method_name':        base_cfg.method_name,
        'model_cfg':          {'in_channels': model_cfg.in_channels,
                               'img_size': model_cfg.img_size,
                               'feature_size': model_cfg.feature_size,
                               'depths': list(model_cfg.depths),
                               'num_heads': list(model_cfg.num_heads)},
    }, model_path)
    logger.info(f'model saved to {model_path}')
else:
    logger.warning('no best_model.pt -- nothing to export')


In [ ]:
logger.info('Done.')
